# KGAT（PyTorch 版）Colab 完整训练 Notebook

这份 notebook 的目标是：**在 Colab 上用 PyTorch 版 KGAT 训练你自己的 `mydata` 数据集**。

## 你要准备的文件
把下面这些文件先放到 Google Drive 的同一个目录中（例如 `MyDrive/KGAT_mydata_source/`）：

- `train.txt`
- `test.txt`
- `user_list.txt`
- `item_list.txt`
- `entity_list.txt`
- `relation_list.txt`
- `kg_final.txt`

可选：
- `item_id_map.json`
- `stats.json`

## 这份 notebook 做什么
1. 挂载 Google Drive  
2. 克隆 KGAT PyTorch 仓库  
3. 把你的 `mydata` 文件复制到仓库需要的 `datasets/mydata/`  
4. 检查数据是否齐全  
5. 用 `main_kgat.py --data_name mydata` 开始训练  
6. 训练后查找模型与结果文件

In [ ]:
# 0. 检查 GPU
import torch, platform, sys
print("Python:", sys.version)
print("Platform:", platform.platform())
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.6.113+-x86_64-with-glibc2.35
Torch: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


In [ ]:
# 1. 挂载 Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 2. 路径设置（你只需要改 SOURCE_DIR）
import os
from pathlib import Path

# 这里改成你在 Drive 里放数据文件的目录
SOURCE_DIR = "/content/drive/MyDrive/KGAT_mydata_source"

# 数据集名字
DATA_NAME = "mydata"

# 仓库目录
REPO_DIR = "/content/KGAT_pytorch"
TARGET_DATA_DIR = f"{REPO_DIR}/datasets/{DATA_NAME}"

# 需要的文件
REQUIRED_FILES = [
    "train.txt",
    "test.txt",
    "user_list.txt",
    "item_list.txt",
    "entity_list.txt",
    "relation_list.txt",
    "kg_final.txt",
]

OPTIONAL_FILES = [
    "item_id_map.json",
    "stats.json",
]

print("SOURCE_DIR =", SOURCE_DIR)
print("REPO_DIR   =", REPO_DIR)
print("TARGET_DIR =", TARGET_DATA_DIR)

SOURCE_DIR = /content/drive/MyDrive/KGAT_mydata_source
REPO_DIR   = /content/KGAT_pytorch
TARGET_DIR = /content/KGAT_pytorch/datasets/mydata


In [ ]:
# 3. 安装基础依赖
# 这里不强行降级 torch，只补常用依赖。
# 如果后面仓库代码因为版本太旧报错，再单独调整。
!pip -q install numpy pandas scipy tqdm scikit-learn

In [ ]:
# 4. 克隆 KGAT PyTorch 仓库
%cd /content
!rm -rf /content/KGAT_pytorch
!git clone https://github.com/LunaBlack/KGAT-pytorch.git KGAT_pytorch
%cd /content/KGAT_pytorch
!ls -lah

/content
Cloning into 'KGAT_pytorch'...
remote: Enumerating objects: 471, done.
remote: Counting objects: 100% (145/145), done.
remote: Compressing objects: 100% (22/22), done.
remote: Total 471 (delta 131), reused 123 (delta 123), pack-reused 326 (from 1)
Receiving objects: 100% (471/471), 106.74 MiB | 80.84 MiB/s, done.
Resolving deltas: 100% (303/303), done.
/content/KGAT_pytorch
total 96K
drwxr-xr-x 8 root root 4.0K Mar 31 13:45 .
drwxr-xr-x 1 root root 4.0K Mar 31 13:45 ..
drwxr-xr-x 2 root root 4.0K Mar 31 13:45 data_loader
drwxr-xr-x 6 root root 4.0K Mar 31 13:45 datasets
drwxr-xr-x 8 root root 4.0K Mar 31 13:45 .git
-rw-r--r-- 1 root root  207 Mar 31 13:45 .gitignore
-rw-r--r-- 1 root root 7.6K Mar 31 13:45 main_bprmf.py
-rw-r--r-- 1 root root 8.2K Mar 31 13:45 main_cke.py
-rw-r--r-- 1 root root 7.8K Mar 31 13:45 main_ecfkg.py
-rw-r--r-- 1 root root 9.9K Mar 31 13:45 main_kgat.py
-rw-r--r-- 1 root root  10K Mar 31 13:45 main_nfm.py
drwxr-xr-x 2 root root 4.0K Mar 31 13:45 model

In [ ]:
# 5. 把你的 mydata 拷贝到仓库需要的 datasets/mydata
import shutil, os
from pathlib import Path

src = Path(SOURCE_DIR)
dst = Path(TARGET_DATA_DIR)
dst.mkdir(parents=True, exist_ok=True)

missing = []
for fn in REQUIRED_FILES:
    src_file = src / fn
    if not src_file.exists():
        missing.append(fn)
    else:
        shutil.copy2(src_file, dst / fn)

for fn in OPTIONAL_FILES:
    src_file = src / fn
    if src_file.exists():
        shutil.copy2(src_file, dst / fn)

if missing:
    raise FileNotFoundError(f"下面这些必要文件在 SOURCE_DIR 里没找到: {missing}")

print("已复制到:", dst)
print(sorted([p.name for p in dst.iterdir()]))

已复制到: /content/KGAT_pytorch/datasets/mydata
['entity_list.txt', 'item_id_map.json', 'item_list.txt', 'kg_final.txt', 'relation_list.txt', 'stats.json', 'test.txt', 'train.txt', 'user_list.txt']


In [ ]:
# 6. 检查 datasets/mydata 目录
!ls -lah /content/KGAT_pytorch/datasets/mydata

total 3.4M
drwxr-xr-x 2 root root 4.0K Mar 31 13:45 .
drwxr-xr-x 7 root root 4.0K Mar 31 13:45 ..
-rw------- 1 root root 313K Mar 30 12:04 entity_list.txt
-rw------- 1 root root  57K Mar 30 12:04 item_id_map.json
-rw------- 1 root root  77K Mar 30 12:04 item_list.txt
-rw------- 1 root root 252K Mar 30 12:04 kg_final.txt
-rw------- 1 root root   64 Mar 30 12:04 relation_list.txt
-rw------- 1 root root  295 Mar 30 12:04 stats.json
-rw------- 1 root root  63K Mar 30 12:04 test.txt
-rw------- 1 root root 2.6M Mar 30 12:04 train.txt
-rw------- 1 root root  63K Mar 30 12:04 user_list.txt


In [ ]:
# 7. 查看数据前几行，确认格式没错
print("===== train.txt =====")
!head -n 3 /content/KGAT_pytorch/datasets/mydata/train.txt

print("\n===== test.txt =====")
!head -n 3 /content/KGAT_pytorch/datasets/mydata/test.txt

print("\n===== relation_list.txt =====")
!cat /content/KGAT_pytorch/datasets/mydata/relation_list.txt

print("\n===== kg_final.txt =====")
!head -n 5 /content/KGAT_pytorch/datasets/mydata/kg_final.txt

===== train.txt =====
1 1 150 260 527 531 588 594 595 608 783 919 938 1022 1028 1029 1035 1097 1193 1207 1246 1270 1287 1545 1566 1721 1836 1961 1962 2018 2028 2294 2355 2398 2692 2762 2791 2797 2804 2918 3105 3114 3186 3408
2 110 163 265 318 349 356 368 380 457 480 515 589 590 593 648 736 920 982 1096 1124 1188 1193 1196 1198 1207 1210 1225 1246 1247 1259 1293 1357 1370 1442 1527 1537 1610 1784 1834 1873 1945 1953 1954 1955 1957 1962 2028 2067 2194 2236 2268 2353 2396 2501 2571 2858 2943 3030 3035 3068 3071 3095 3105 3147 3255 3334 3418 3451 3468 3471 3578
3 260 480 552 590 653 733 1049 1079 1136 1196 1197 1198 1210 1259 1266 1291 1304 1378 1379 1394 1615 1961 1968 2006 2115 2167 2355 2470 2735 2858 2871 3168 3421 3552 3671

===== test.txt =====
1 1907
2 2002
3 104

===== relation_list.txt =====
freebase_id remap_id
DIRECTED_BY 0
HAS_GENRE 1
STARRED_BY 2

===== kg_final.txt =====
0 1 3531
0 1 3532
0 1 3533
1 1 3530
1 1 3532


## 开始训练

这版 notebook 会同时保存：

- `best_model.pth`：验证指标最好的模型
- `last_model.pth`：训练结束时最后一轮模型
- `best_metrics.json`：最佳模型对应指标
- `metrics_history.json`：每次评估的指标历史
- `embeddings/best_user_embeddings.npy`
- `embeddings/best_item_embeddings.npy`
- `embeddings/last_user_embeddings.npy`
- `embeddings/last_item_embeddings.npy`

其中默认的：

- `embeddings/user_embeddings.npy`
- `embeddings/item_embeddings.npy`

也会指向 **best model 对应的 embedding**。


In [ ]:
from pathlib import Path

file_path = Path("/content/KGAT_pytorch/data_loader/loader_base.py")
text = file_path.read_text(encoding="utf-8")

replacements = {
    "batch_user = random.sample(exist_users, batch_size)": "batch_user = random.sample(sorted(exist_users), batch_size)",
    "batch_head = random.sample(exist_heads, batch_size)": "batch_head = random.sample(sorted(exist_heads), batch_size)",
}

changed = False
for old, new in replacements.items():
    if old in text:
        text = text.replace(old, new)
        changed = True

file_path.write_text(text, encoding="utf-8")

if changed:
    print("loader_base.py 已修正")
else:
    print("沒有找到對應舊語句，請檢查 loader_base.py 內容")

沒有找到對應舊語句，請檢查 loader_base.py 內容


In [ ]:
!grep -n "random.sample(sorted(exist_users), batch_size)\|random.sample(sorted(exist_heads), batch_size)" /content/KGAT_pytorch/data_loader/loader_base.py

102:            batch_user = random.sample(sorted(exist_users), batch_size)
153:            batch_head = random.sample(sorted(exist_heads), batch_size)


In [ ]:
from pathlib import Path

# =========================
# 1) patch loader_base.py
# =========================
loader_base_path = Path("/content/KGAT_pytorch/data_loader/loader_base.py")
loader_base_text = loader_base_path.read_text(encoding="utf-8")

loader_base_text = loader_base_text.replace(
    "batch_user = random.sample(exist_users, batch_size)",
    "batch_user = random.sample(sorted(exist_users), batch_size)"
)

loader_base_path.write_text(loader_base_text, encoding="utf-8")
print("Patched loader_base.py")

# =========================
# 2) overwrite main_kgat.py
# =========================
main_path = Path("/content/KGAT_pytorch/main_kgat.py")
main_code = r'''import os
import json
import sys
import random
from time import time

import numpy as np
import torch
import torch.optim as optim
from tqdm import tqdm

from model.KGAT import KGAT
from parser.parser_kgat import *
from utils.log_helper import *
from utils.metrics import *
from utils.model_helper import *
from data_loader.loader_kgat import DataLoaderKGAT


def evaluate(model, dataloader, Ks, device):
    test_batch_size = dataloader.test_batch_size
    train_user_dict = dataloader.train_user_dict
    test_user_dict = dataloader.test_user_dict

    model.eval()

    user_ids = list(test_user_dict.keys())
    user_ids_batches = [user_ids[i: i + test_batch_size] for i in range(0, len(user_ids), test_batch_size)]
    user_ids_batches = [torch.LongTensor(d) for d in user_ids_batches]

    n_items = dataloader.n_items
    item_ids = torch.arange(n_items, dtype=torch.long).to(device)

    cf_scores = []
    metric_names = ['precision', 'recall', 'ndcg']
    metrics_dict = {k: {m: [] for m in metric_names} for k in Ks}

    with tqdm(total=len(user_ids_batches), desc='Evaluating Iteration') as pbar:
        for batch_user_ids in user_ids_batches:
            batch_user_ids = batch_user_ids.to(device)

            with torch.no_grad():
                batch_scores = model(batch_user_ids, item_ids, mode='predict')

            batch_scores = batch_scores.cpu()
            batch_metrics = calc_metrics_at_k(
                batch_scores,
                train_user_dict,
                test_user_dict,
                batch_user_ids.cpu().numpy(),
                item_ids.cpu().numpy(),
                Ks,
            )

            cf_scores.append(batch_scores.numpy())
            for k in Ks:
                for m in metric_names:
                    metrics_dict[k][m].append(batch_metrics[k][m])
            pbar.update(1)

    cf_scores = np.concatenate(cf_scores, axis=0)
    for k in Ks:
        for m in metric_names:
            metrics_dict[k][m] = np.concatenate(metrics_dict[k][m]).mean()
    return cf_scores, metrics_dict


def save_embeddings_from_state_dict(state_dict, data, save_embeddings_dir, prefix=None, write_default_names=False):
    os.makedirs(save_embeddings_dir, exist_ok=True)

    all_embeddings = state_dict['entity_user_embed.weight']
    if hasattr(all_embeddings, 'detach'):
        all_embeddings = all_embeddings.detach().cpu().numpy()
    else:
        all_embeddings = np.asarray(all_embeddings)

    # loader_kgat.py 的 ID 体系：
    # 前面是 entities/items，users 从 n_entities 开始
    item_embeddings = all_embeddings[0:data.n_items]
    user_embeddings = all_embeddings[data.n_entities : data.n_entities + data.n_users]

    if prefix:
        user_path = os.path.join(save_embeddings_dir, f'{prefix}_user_embeddings.npy')
        item_path = os.path.join(save_embeddings_dir, f'{prefix}_item_embeddings.npy')
        np.save(user_path, user_embeddings)
        np.save(item_path, item_embeddings)
        print(f'Saved {prefix} user embeddings to: {user_path}')
        print(f'Saved {prefix} item embeddings to: {item_path}')

    if write_default_names:
        user_path = os.path.join(save_embeddings_dir, 'user_embeddings.npy')
        item_path = os.path.join(save_embeddings_dir, 'item_embeddings.npy')
        np.save(user_path, user_embeddings)
        np.save(item_path, item_embeddings)
        print(f'Saved default user embeddings to: {user_path}')
        print(f'Saved default item embeddings to: {item_path}')


def train(args):
    random.seed(args.seed)
    np.random.seed(args.seed)
    torch.manual_seed(args.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(args.seed)

    os.makedirs(args.save_dir, exist_ok=True)

    log_save_id = create_log_id(args.save_dir)
    logging_config(folder=args.save_dir, name='log{:d}'.format(log_save_id), no_console=False)
    logging.info(args)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    data = DataLoaderKGAT(args, logging)
    if args.use_pretrain == 1:
        user_pre_embed = torch.tensor(data.user_pre_embed)
        item_pre_embed = torch.tensor(data.item_pre_embed)
    else:
        user_pre_embed, item_pre_embed = None, None

    model = KGAT(args, data.n_users, data.n_entities, data.n_relations, data.A_in, user_pre_embed, item_pre_embed)
    if args.use_pretrain == 2:
        model = load_model(model, args.pretrain_model_path)

    model.to(device)
    logging.info(model)

    cf_optimizer = optim.Adam(model.parameters(), lr=args.lr)
    kg_optimizer = optim.Adam(model.parameters(), lr=args.lr)

    Ks = eval(args.Ks)
    k_min = min(Ks)
    k_max = max(Ks)

    epoch_list = []
    metrics_list = {k: {'precision': [], 'recall': [], 'ndcg': []} for k in Ks}

    best_epoch = -1
    best_recall = -1.0
    best_metrics = None

    best_model_path = os.path.join(args.save_dir, 'best_model.pth')
    last_model_path = os.path.join(args.save_dir, 'last_model.pth')
    best_metrics_path = os.path.join(args.save_dir, 'best_metrics.json')
    metrics_history_path = os.path.join(args.save_dir, 'metrics_history.json')
    save_embeddings_dir = os.path.join(args.save_dir, 'embeddings')

    for epoch in range(1, args.n_epoch + 1):
        time0 = time()
        model.train()

        # =========================
        # CF training
        # =========================
        time1 = time()
        cf_total_loss = 0.0
        n_cf_batch = data.n_cf_train // data.cf_batch_size + 1

        for iter in range(1, n_cf_batch + 1):
            time2 = time()
            cf_batch_user, cf_batch_pos_item, cf_batch_neg_item = data.generate_cf_batch(
                data.train_user_dict, data.cf_batch_size
            )
            cf_batch_user = cf_batch_user.to(device)
            cf_batch_pos_item = cf_batch_pos_item.to(device)
            cf_batch_neg_item = cf_batch_neg_item.to(device)

            cf_batch_loss = model(cf_batch_user, cf_batch_pos_item, cf_batch_neg_item, mode='train_cf')
            if np.isnan(cf_batch_loss.detach().cpu().numpy()):
                logging.info('ERROR: CF training loss is nan.')
                sys.exit()

            cf_optimizer.zero_grad()
            cf_batch_loss.backward()
            cf_optimizer.step()

            cf_total_loss += cf_batch_loss.item()

            if (iter % args.cf_print_every) == 0:
                logging.info(
                    'CF Training: Epoch {:04d} Iter {:04d} / {:04d} | Time {:.1f}s | Iter Loss {:.4f} | Iter Mean Loss {:.4f}'.format(
                        epoch, iter, n_cf_batch, time() - time2, cf_batch_loss.item(), cf_total_loss / iter
                    )
                )

        logging.info(
            'CF Training: Epoch {:04d} [{:.1f}s] | Total Mean Loss {:.4f}'.format(
                epoch, time() - time1, cf_total_loss / max(1, n_cf_batch)
            )
        )

        # =========================
        # KG training
        # =========================
        time3 = time()
        kg_total_loss = 0.0
        n_kg_batch = data.n_kg_train // data.kg_batch_size + 1

        for iter in range(1, n_kg_batch + 1):
            time2 = time()
            kg_batch_head, kg_batch_relation, kg_batch_pos_tail, kg_batch_neg_tail = data.generate_kg_batch(
                data.train_kg_dict, data.kg_batch_size, data.n_users_entities
            )
            kg_batch_head = kg_batch_head.to(device)
            kg_batch_relation = kg_batch_relation.to(device)
            kg_batch_pos_tail = kg_batch_pos_tail.to(device)
            kg_batch_neg_tail = kg_batch_neg_tail.to(device)

            kg_batch_loss = model(
                kg_batch_head, kg_batch_relation, kg_batch_pos_tail, kg_batch_neg_tail, mode='train_kg'
            )
            if np.isnan(kg_batch_loss.detach().cpu().numpy()):
                logging.info('ERROR: KG training loss is nan.')
                sys.exit()

            kg_optimizer.zero_grad()
            kg_batch_loss.backward()
            kg_optimizer.step()

            kg_total_loss += kg_batch_loss.item()

            if (iter % args.kg_print_every) == 0:
                logging.info(
                    'KG Training: Epoch {:04d} Iter {:04d} / {:04d} | Time {:.1f}s | Iter Loss {:.4f} | Iter Mean Loss {:.4f}'.format(
                        epoch, iter, n_kg_batch, time() - time2, kg_batch_loss.item(), kg_total_loss / iter
                    )
                )

        logging.info(
            'KG Training: Epoch {:04d} [{:.1f}s] | Total Mean Loss {:.4f}'.format(
                epoch, time() - time3, kg_total_loss / max(1, n_kg_batch)
            )
        )

        # =========================
        # Update attention
        # =========================
        time5 = time()
        h_list = data.h_list.to(device)
        t_list = data.t_list.to(device)
        r_list = data.r_list.to(device)
        relations = list(data.laplacian_dict.keys())
        model(h_list, t_list, r_list, relations, mode='update_att')
        logging.info('Update Attention: Epoch {:04d} | Total Time {:.1f}s'.format(epoch, time() - time5))

        # =========================
        # Evaluate
        # =========================
        if (epoch % args.evaluate_every) == 0:
            time4 = time()
            _, metrics_dict = evaluate(model, data, Ks, device)

            ret = ''
            for k in Ks:
                ret += 'recall@{}: {:.4f}    precision@{}: {:.4f}    ndcg@{}: {:.4f}    '.format(
                    k, metrics_dict[k]['recall'], k, metrics_dict[k]['precision'], k, metrics_dict[k]['ndcg']
                )
            logging.info('Evaluation: Epoch {:04d} [{:.1f}s] | {}'.format(epoch, time() - time4, ret.strip()))

            epoch_list.append(epoch)
            for k in Ks:
                metrics_list[k]['precision'].append(float(metrics_dict[k]['precision']))
                metrics_list[k]['recall'].append(float(metrics_dict[k]['recall']))
                metrics_list[k]['ndcg'].append(float(metrics_dict[k]['ndcg']))

            current_recall = float(metrics_dict[k_min]['recall'])
            if current_recall > best_recall:
                best_recall = current_recall
                best_epoch = epoch
                best_metrics = {
                    'epoch_idx': int(epoch),
                    'monitor_k': int(k_min),
                }
                for k in Ks:
                    best_metrics[f'recall@{k}'] = float(metrics_dict[k]['recall'])
                    best_metrics[f'precision@{k}'] = float(metrics_dict[k]['precision'])
                    best_metrics[f'ndcg@{k}'] = float(metrics_dict[k]['ndcg'])

                torch.save(model.state_dict(), best_model_path)
                logging.info('Saved new best model to {}'.format(best_model_path))

            _, should_stop = early_stopping(metrics_list[k_min]['recall'], args.stopping_steps)
            if should_stop:
                logging.info(
                    'Early stopping triggered at epoch {:04d}. Best epoch so far: {:04d}'.format(
                        epoch, best_epoch if best_epoch >= 0 else epoch
                    )
                )
                break

        logging.info('Epoch {:04d} finished in {:.1f}s'.format(epoch, time() - time0))

    # =========================
    # Save last model
    # =========================
    torch.save(model.state_dict(), last_model_path)
    logging.info('Saved last model to {}'.format(last_model_path))

    if best_metrics is None:
        best_metrics = {
            'epoch_idx': -1,
            'monitor_k': int(k_min),
            'note': 'No evaluation was run before training ended; best_model is copied from last_model.'
        }
        for k in Ks:
            best_metrics[f'recall@{k}'] = 0.0
            best_metrics[f'precision@{k}'] = 0.0
            best_metrics[f'ndcg@{k}'] = 0.0
        torch.save(model.state_dict(), best_model_path)
        logging.info('No evaluation checkpoint found. Copied last model to {}'.format(best_model_path))

    with open(best_metrics_path, 'w', encoding='utf-8') as f:
        json.dump(best_metrics, f, ensure_ascii=False, indent=2)
    logging.info('Saved best metrics to {}'.format(best_metrics_path))

    history_payload = {
        'epoch_list': epoch_list,
        'metrics': metrics_list,
        'best_epoch': int(best_metrics.get('epoch_idx', -1)),
    }
    with open(metrics_history_path, 'w', encoding='utf-8') as f:
        json.dump(history_payload, f, ensure_ascii=False, indent=2)
    logging.info('Saved metrics history to {}'.format(metrics_history_path))

    save_embeddings_from_state_dict(
        model.state_dict(),
        data,
        save_embeddings_dir,
        prefix='last',
        write_default_names=False,
    )
    logging.info('Saved last-model embeddings to {}'.format(save_embeddings_dir))

    best_state_dict = torch.load(best_model_path, map_location='cpu')
    save_embeddings_from_state_dict(
        best_state_dict,
        data,
        save_embeddings_dir,
        prefix='best',
        write_default_names=True,
    )
    logging.info('Saved best-model embeddings to {}'.format(save_embeddings_dir))

    logging.info(
        'best_epoch: {:d} | recall@{:d}: {:.4f}, recall@{:d}: {:.4f} | precision@{:d}: {:.4f}, precision@{:d}: {:.4f} | ndcg@{:d}: {:.4f}, ndcg@{:d}: {:.4f}'.format(
            int(best_metrics.get('epoch_idx', -1)),
            k_min, float(best_metrics.get(f'recall@{k_min}', 0.0)),
            k_max, float(best_metrics.get(f'recall@{k_max}', 0.0)),
            k_min, float(best_metrics.get(f'precision@{k_min}', 0.0)),
            k_max, float(best_metrics.get(f'precision@{k_max}', 0.0)),
            k_min, float(best_metrics.get(f'ndcg@{k_min}', 0.0)),
            k_max, float(best_metrics.get(f'ndcg@{k_max}', 0.0)),
        )
    )


if __name__ == '__main__':
    args = parse_kgat_args()
    train(args)
'''
main_path.write_text(main_code, encoding="utf-8")
print("Overwrote main_kgat.py")

Patched loader_base.py
Overwrote main_kgat.py


In [ ]:
!grep -n "sorted(exist_users)\|train_kg_dict\|generate_kg_batch\|kg_batch_pos_tail\|kg_batch_neg_tail\|update_att\|item_embeddings =\|user_embeddings =" /content/KGAT_pytorch/data_loader/loader_base.py /content/KGAT_pytorch/main_kgat.py

/content/KGAT_pytorch/data_loader/loader_base.py:102:            batch_user = random.sample(sorted(exist_users), batch_size)
/content/KGAT_pytorch/data_loader/loader_base.py:150:    def generate_kg_batch(self, kg_dict, batch_size, highest_neg_idx):
/content/KGAT_pytorch/main_kgat.py:79:    item_embeddings = all_embeddings[0:data.n_items]
/content/KGAT_pytorch/main_kgat.py:80:    user_embeddings = all_embeddings[data.n_entities : data.n_entities + data.n_users]
/content/KGAT_pytorch/main_kgat.py:201:            kg_batch_head, kg_batch_relation, kg_batch_pos_tail, kg_batch_neg_tail = data.generate_kg_batch(
/content/KGAT_pytorch/main_kgat.py:202:                data.train_kg_dict, data.kg_batch_size, data.n_users_entities
/content/KGAT_pytorch/main_kgat.py:206:            kg_batch_pos_tail = kg_batch_pos_tail.to(device)
/content/KGAT_pytorch/main_kgat.py:207:            kg_batch_neg_tail = kg_batch_neg_tail.to(device)
/content/KGAT_pytorch/main_kgat.py:210:                kg_batch_head, 

In [ ]:
# 8. 开始训练 KGAT（先用最小可运行命令）
%cd /content/KGAT_pytorch
!python main_kgat.py --data_name mydata --use_pretrain 0

串流輸出內容已截斷至最後 5000 行。
2026-03-31 14:46:05,180 - root - INFO - KG Training: Epoch 0107 Iter 0297 / 0570 | Time 0.0s | Iter Loss 0.0240 | Iter Mean Loss 0.0243
2026-03-31 14:46:05,208 - root - INFO - KG Training: Epoch 0107 Iter 0298 / 0570 | Time 0.0s | Iter Loss 0.0250 | Iter Mean Loss 0.0243
2026-03-31 14:46:05,238 - root - INFO - KG Training: Epoch 0107 Iter 0299 / 0570 | Time 0.0s | Iter Loss 0.0235 | Iter Mean Loss 0.0243
2026-03-31 14:46:05,268 - root - INFO - KG Training: Epoch 0107 Iter 0300 / 0570 | Time 0.0s | Iter Loss 0.0213 | Iter Mean Loss 0.0243
2026-03-31 14:46:05,297 - root - INFO - KG Training: Epoch 0107 Iter 0301 / 0570 | Time 0.0s | Iter Loss 0.0207 | Iter Mean Loss 0.0243
2026-03-31 14:46:05,327 - root - INFO - KG Training: Epoch 0107 Iter 0302 / 0570 | Time 0.0s | Iter Loss 0.0321 | Iter Mean Loss 0.0243
2026-03-31 14:46:05,356 - root - INFO - KG Training: Epoch 0107 Iter 0303 / 0570 | Time 0.0s | Iter Loss 0.0226 | Iter Mean Loss 0.0243
2026-03-31 14:46:05,387 - r

In [ ]:
# 9. 训练后查找模型、指标、embedding、日志
%cd /content/KGAT_pytorch
print("===== 可能的模型/日志/结果文件 =====")
!find /content/KGAT_pytorch -type f | grep -E "best_model\.pth$|last_model\.pth$|model\.pth$|best_metrics\.json$|metrics_history\.json$|user_embeddings\.npy$|item_embeddings\.npy$|log[0-9]+\.log$" | sort


/content/KGAT_pytorch
===== 可能的模型/日志/结果文件 =====
/content/KGAT_pytorch/trained_model/KGAT/mydata/embed-dim64_relation-dim64_random-walk_bi-interaction_64-32-16_lr0.0001_pretrain0/best_model.pth
/content/KGAT_pytorch/trained_model/KGAT/mydata/embed-dim64_relation-dim64_random-walk_bi-interaction_64-32-16_lr0.0001_pretrain0/log0.log
/content/KGAT_pytorch/trained_model/KGAT/mydata/embed-dim64_relation-dim64_random-walk_bi-interaction_64-32-16_lr0.0001_pretrain0/log1.log
/content/KGAT_pytorch/trained_model/KGAT/mydata/embed-dim64_relation-dim64_random-walk_bi-interaction_64-32-16_lr0.0001_pretrain0/log2.log
/content/KGAT_pytorch/trained_model/KGAT/mydata/embed-dim64_relation-dim64_random-walk_bi-interaction_64-32-16_lr0.0001_pretrain0/log3.log


In [ ]:
import os
import numpy as np
import torch

from parser.parser_kgat import parse_kgat_args
from data_loader.loader_kgat import DataLoaderKGAT

# 1) 路径改成你的 best_model.pth 实际位置
best_model_path = "/content/KGAT_pytorch/trained_model/KGAT/mydata/embed-dim64_relation-dim64_random-walk_bi-interaction_64-32-16_lr0.0001_pretrain0/best_model.pth"

# 2) 输出目录
out_dir = "/content/KGAT_pytorch/exported_best_embeddings"
os.makedirs(out_dir, exist_ok=True)

# 3) 读取数据配置
args = parse_kgat_args()
args.data_name = "mydata"
args.use_pretrain = 0

data = DataLoaderKGAT(args, logging=__import__("logging"))

# 4) 读 checkpoint
state_dict = torch.load(best_model_path, map_location="cpu")

# 5) 取总 embedding 表
all_embeddings = state_dict["entity_user_embed.weight"].detach().cpu().numpy()

# 6) 按你现在 loader_kgat.py 的 ID 体系切开
# 前面是 entities/items，users 从 n_entities 开始
item_embeddings = all_embeddings[0:data.n_items]
user_embeddings = all_embeddings[data.n_entities : data.n_entities + data.n_users]

# 7) 保存
np.save(os.path.join(out_dir, "best_item_embeddings.npy"), item_embeddings)
np.save(os.path.join(out_dir, "best_user_embeddings.npy"), user_embeddings)

# 也顺手保存默认名字，后面脚本更方便直接读
np.save(os.path.join(out_dir, "item_embeddings.npy"), item_embeddings)
np.save(os.path.join(out_dir, "user_embeddings.npy"), user_embeddings)

print("导出完成：")
print(os.path.join(out_dir, "best_item_embeddings.npy"))
print(os.path.join(out_dir, "best_user_embeddings.npy"))
print("item_embeddings shape =", item_embeddings.shape)
print("user_embeddings shape =", user_embeddings.shape)

usage: colab_kernel_launcher.py [-h] [--seed SEED] [--data_name [DATA_NAME]]
                                [--data_dir [DATA_DIR]]
                                [--use_pretrain USE_PRETRAIN]
                                [--pretrain_embedding_dir [PRETRAIN_EMBEDDING_DIR]]
                                [--pretrain_model_path [PRETRAIN_MODEL_PATH]]
                                [--cf_batch_size CF_BATCH_SIZE]
                                [--kg_batch_size KG_BATCH_SIZE]
                                [--test_batch_size TEST_BATCH_SIZE]
                                [--embed_dim EMBED_DIM]
                                [--relation_dim RELATION_DIM]
                                [--laplacian_type LAPLACIAN_TYPE]
                                [--aggregation_type AGGREGATION_TYPE]
                                [--conv_dim_list [CONV_DIM_LIST]]
                                [--mess_dropout [MESS_DROPOUT]]
                                [--kg_l2loss_lambda KG_L2LOSS_L

SystemExit: 2

In [ ]:
# 10. 快速查看 best model 的指标
import json
from pathlib import Path

root = Path("/content/KGAT_pytorch/trained_model/KGAT")
files = sorted(root.rglob("best_metrics.json"))
if not files:
    print("还没有找到 best_metrics.json，请先确认训练已经跑完。")
else:
    latest = files[-1]
    print("Using:", latest)
    print(latest.read_text(encoding="utf-8"))


还没有找到 best_metrics.json，请先确认训练已经跑完。


In [ ]:
# 10. 如果你只想看最近改动过的文件，可以跑这个
%cd /content/KGAT_pytorch
!find . -type f -printf "%TY-%Tm-%Td %TH:%TM:%TS %p\n" | sort | tail -n 100

/content/KGAT_pytorch
2026-03-31 08:46:30.4361282640 ./.git/hooks/fsmonitor-watchman.sample
2026-03-31 08:46:30.4361282640 ./.git/hooks/post-update.sample
2026-03-31 08:46:30.4361282640 ./.git/hooks/pre-applypatch.sample
2026-03-31 08:46:30.4361282640 ./.git/hooks/pre-commit.sample
2026-03-31 08:46:30.4361282640 ./.git/hooks/pre-merge-commit.sample
2026-03-31 08:46:30.4361282640 ./.git/hooks/prepare-commit-msg.sample
2026-03-31 08:46:30.4361282640 ./.git/hooks/pre-push.sample
2026-03-31 08:46:30.4361282640 ./.git/hooks/pre-rebase.sample
2026-03-31 08:46:30.4361282640 ./.git/hooks/pre-receive.sample
2026-03-31 08:46:30.4361282640 ./.git/hooks/push-to-checkout.sample
2026-03-31 08:46:30.4361282640 ./.git/hooks/update.sample
2026-03-31 08:46:30.4361282640 ./.git/info/exclude
2026-03-31 08:46:33.4083841100 ./.git/objects/pack/pack-a91323a5305544199aae4d352b8707042795dc4a.idx
2026-03-31 08:46:33.4083841100 ./.git/objects/pack/pack-a91323a5305544199aae4d352b8707042795dc4a.pack
2026-03-31 08:

## 可选：如果默认训练能跑通，再试第二轮

你可以复制上一格，把命令改成更长一点的形式，比如：

```bash
python main_kgat.py --data_name mydata
```

由于不同 KGAT PyTorch 复现仓库支持的参数名不完全一样，  
这份 notebook 先不强行写一长串参数，避免一开始就因为参数名不匹配而报错。

更稳的做法是：
1. 先用默认命令跑通  
2. 看训练日志  
3. 再决定要不要加参数

## 如果训练报错，优先检查这几件事

### 1. 数据路径
确保你的数据真的在：

`/content/KGAT_pytorch/datasets/mydata/`

### 2. 文件格式
- `train.txt` / `test.txt`：每行是 `user item1 item2 ...`
- `kg_final.txt`：每行是 `head relation tail`
- `user_list.txt` / `item_list.txt` / `entity_list.txt` / `relation_list.txt`：是映射表

### 3. id 是否一致
最常见的问题是：
- `train.txt / test.txt` 里的 item id 还在用原始 movie id
- 但 `kg_final.txt` / `item_list.txt` 用的是 remap id

这会导致训练直接错位。

### 4. 版本兼容
这个 KGAT PyTorch 仓库 README 里写的测试环境比较旧。  
如果默认 Colab 环境报依赖兼容错误，再根据报错去补版本。